# Lab 15: The Messages List In Depth / GroqClient Wrapper Class

## The capstone

This lab wraps the hand-rolled `messages` list from lab 13 into a `GroqClient` class that owns `self.messages` and grows it automatically inside `send()`. The constructor also takes a `transport` callable (default: a real API call), which is what lets the exercise below test the class with `fake_transport` and zero network access.

## Exercise

Implement `GroqClient`: `__init__` stores `api_key`, `model`, an optional `system_prompt` (seeded as the first message if given), and a `transport` callable (signature: `transport(messages, api_key, model) -> str`). `add_user_message` and `add_assistant_message` append to `self.messages`. `send(user_text)` adds the user message, calls the transport with the *current* messages, adds the reply as an assistant message, and returns the reply text. `reset()` clears back to just the system prompt (if any). `history` is a read-only property returning a copy of `self.messages`.

In [ ]:
class GroqClient:
    def __init__(self, api_key: str | None, model: str, system_prompt: str | None = None, transport=None):
        self.api_key = api_key
        self.model = model
        self.system_prompt = system_prompt
        self.transport = transport
        self.messages: list[dict] = []
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})

    def add_user_message(self, text: str) -> None:
        # TODO: append a {"role": "user", "content": text} message.
        raise NotImplementedError("GroqClient.add_user_message() is not implemented yet - replace the TODO with your code")

    def add_assistant_message(self, text: str) -> None:
        # TODO: append a {"role": "assistant", "content": text} message.
        raise NotImplementedError("GroqClient.add_assistant_message() is not implemented yet - replace the TODO with your code")

    def send(self, user_text: str) -> str:
        """TODO: add the user message, call self.transport(self.messages,
        self.api_key, self.model) to get the reply text, add it as an
        assistant message, and return it."""
        raise NotImplementedError("GroqClient.send() is not implemented yet - replace the TODO with your code")

    def reset(self) -> None:
        """TODO: clear self.messages back down to just the system prompt
        (if self.system_prompt was set), otherwise an empty list."""
        raise NotImplementedError("GroqClient.reset() is not implemented yet - replace the TODO with your code")

    @property
    def history(self) -> list[dict]:
        return list(self.messages)


def fake_transport(messages: list[dict], api_key: str | None, model: str) -> str:
    """A stand-in for a real API call - lets us test GroqClient with zero network access."""
    last_user_message = messages[-1]["content"]
    return f"(fake reply to: {last_user_message})"


client = GroqClient(api_key=None, model="fake-model", system_prompt="You are helpful.", transport=fake_transport)
assert client.history == [{"role": "system", "content": "You are helpful."}], 'client.history should be ' + repr([{"role": "system", "content": "You are helpful."}]) + ", got " + repr(client.history)

reply = client.send("Hello!")
assert reply == "(fake reply to: Hello!)", 'reply should be ' + repr("(fake reply to: Hello!)") + ", got " + repr(reply)
assert client.history == [
    {"role": "system", "content": "You are helpful."},
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": reply},
], 'client.history should be ' + repr([
    {"role": "system", "content": "You are helpful."},
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": reply},
]) + ", got " + repr(client.history)

client.send("Second message")
assert len(client.history) == 5, 'len(client.history) should be ' + repr(5) + ", got " + repr(len(client.history))

client.reset()
assert client.history == [{"role": "system", "content": "You are helpful."}], 'client.history should be ' + repr([{"role": "system", "content": "You are helpful."}]) + ", got " + repr(client.history)

print("All checks passed - GroqClient's message-list logic works with zero network access.")

## Solution

In [ ]:
class GroqClient:
    def __init__(self, api_key: str | None, model: str, system_prompt: str | None = None, transport=None):
        self.api_key = api_key
        self.model = model
        self.system_prompt = system_prompt
        self.transport = transport
        self.messages: list[dict] = []
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})

    def add_user_message(self, text: str) -> None:
        self.messages.append({"role": "user", "content": text})

    def add_assistant_message(self, text: str) -> None:
        self.messages.append({"role": "assistant", "content": text})

    def send(self, user_text: str) -> str:
        self.add_user_message(user_text)
        reply = self.transport(self.messages, self.api_key, self.model)
        self.add_assistant_message(reply)
        return reply

    def reset(self) -> None:
        self.messages = [{"role": "system", "content": self.system_prompt}] if self.system_prompt else []

    @property
    def history(self) -> list[dict]:
        return list(self.messages)


client = GroqClient(api_key=None, model="fake-model", system_prompt="You are helpful.", transport=fake_transport)
print(client.send("Hello!"))

## Real-world project (LIVE - needs your GROQ_API_KEY above)

Rebuilds lab 13's digital twin on top of `GroqClient` instead of a hand-rolled messages list, using a real transport this time. Fill in `my_profile` with your own bio, same as lab 13.

In [ ]:
"""Get your free Groq API key at https://console.groq.com/keys - it's never written to this notebook."""
import getpass
import json
import requests

GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
DEFAULT_MODEL = "llama-3.1-8b-instant"

GROQ_API_KEY = getpass.getpass("Enter your GROQ_API_KEY: ")


def build_request_payload(messages, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The request BODY: what we're asking the model to do."""
    payload = {"model": model, "messages": messages, "temperature": temperature}
    if response_format is not None:
        payload["response_format"] = response_format
    return payload


def extract_reply(response_json):
    """Pulls the assistant's text out of the RESPONSE body."""
    return response_json["choices"][0]["message"]["content"]


def call_groq(messages, api_key=None, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The full request: ENDPOINT + METHOD (POST) + HEADERS (auth) + BODY -> RESPONSE."""
    headers = {
        "Authorization": f"Bearer {api_key or GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = build_request_payload(messages, model=model, temperature=temperature, response_format=response_format)
    response = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    return extract_reply(response.json())


print("Groq setup ready. Call call_groq(messages) to send a chat completion request.")


In [ ]:
def build_digital_twin_system_prompt(profile: dict) -> str:
    lines = [
        f"You are role-playing as {profile['name']}, {profile['role']}.",
        f"Speak in a tone that is {profile['tone']}.",
        "Facts you can draw on about this person:",
    ]
    lines += [f"- {fact}" for fact in profile["facts"]]
    lines.append("Boundaries you must respect:")
    lines += [f"- {b}" for b in profile["boundaries"]]
    return "\n".join(lines)


def real_transport(messages: list[dict], api_key: str | None, model: str) -> str:
    return call_groq(messages, api_key=api_key, model=model)


my_profile = {
    "name": "your name",
    "role": "however you'd describe your role in one clause",
    "tone": "a few adjectives for how you talk",
    "facts": ["a real fact about you", "another real fact about you"],
    "boundaries": ["something this digital twin should always refuse to do"],
}

twin = GroqClient(
    api_key=GROQ_API_KEY,
    model=DEFAULT_MODEL,
    system_prompt=build_digital_twin_system_prompt(my_profile),
    transport=real_transport,
)

for user_message in ["Hi! What are you working on lately?", "What's the most useful thing you've learned recently?"]:
    reply = twin.send(user_message)
    print(f"You: {user_message}\nTwin: {reply}\n")

print("--- Full messages list ---")
for message in twin.history:
    print(f"{message['role']:<10} {message['content'][:80]}")